# TP M2-3 — Décoder une trame Bluetooth « Heart Rate Measurement » (0x2A37)

**Module 2 — vendredi 9 octobre 2026 — TP 3 (15:30-16:15)** — MBA Santé, IA & IoT — Epitech — Nicolas LAURIO

Trames **fictives**, usage **pédagogique**, aucune finalité diagnostique. Ce notebook ne télécharge rien et n'utilise aucune donnée personnelle.

**Comment l'ouvrir**
- **(A) Google Colab** : téléchargez ce fichier `.ipynb` depuis Teams / Moodle → https://colab.research.google.com → *Fichier → Importer un notebook*. Puis *Exécution → Tout exécuter*.
- **(B) En local** : Python 3.10+, `pip install numpy scipy pandas matplotlib jupyter`, puis `jupyter notebook` dans le dossier du fichier.

**Déroulé**
1. Vous décodez d'abord **3 trames à la main**, sur papier : les trames 3, 5 et 2 (énoncé `TP_M2_3_BLE_nRFConnect.md`, étape 4).
2. Puis vous vérifiez vos réponses ici. Le notebook décode les **6** trames : il applique la norme, sans se tromper.

**Repères dans ce notebook**
- **▶ Exécutez** : lancez la cellule (bouton ▶ ou Maj + Entrée), sans rien changer.
- **✏️ À vous** : vous modifiez quelque chose dans la cellule, puis vous l'exécutez.
- **Pas besoin de lire ce code** : la cellule fabrique un outil ; seul le résultat compte.
- Les bonus facultatifs sont dans le notebook séparé `bonus/TP_M2_3_BLE_bonus.ipynb`.

Une **trame** est le message envoyé par le capteur. Ici, c'est le message « fréquence cardiaque » du Bluetooth basse consommation (BLE).

**Équipe 02 — voie A.** Les exercices du cours restent fictifs ; le notebook séparé des bonus contient cinq trames réelles de la Garmin Forerunner 265, sans nom du porteur ni adresse Bluetooth.


## 1. Rappel du format (Bluetooth SIG, *GATT Specification Supplement*, § 3.126)

**Le message, case par case.** Chaque case est un **octet** : une case de 8 interrupteurs (bits), chacun à 0 ou 1. On l'écrit avec 2 caractères en **hexadécimal** (16 chiffres : 0 à 9 puis A à F). Exemple : `48` = 4 × 16 + 8 = 72.

| Octet(s) | Ce que c'est | Présent quand ? |
|---|---|---|
| 1 | **Flags** (drapeaux) : 8 interrupteurs oui / non | toujours |
| 1 ou 2 | Fréquence cardiaque (FC), en battements par minute (bpm) | toujours : 1 octet (`uint8`) si interrupteur 0 = 0, 2 octets (`uint16`) si interrupteur 0 = 1 |
| 2 | Énergie dépensée (kJ, cumulée) | interrupteur 3 = 1 |
| 2 × n | Intervalles RR (temps entre deux battements), en 1/1024 de seconde | interrupteur 4 = 1 |

**Les interrupteurs du premier octet** (numérotés de **droite à gauche** : le bit 0 est le plus à droite) :

| Interrupteur (bit) | Question à laquelle il répond (1 = oui, 0 = non) |
|---|---|
| 0 | La FC prend-elle 2 octets ? (0 = 1 octet `uint8`, 1 = 2 octets `uint16`) |
| 1 | Le capteur touche-t-il bien la peau ? (contact **détecté**) |
| 2 | Le capteur sait-il vérifier le contact ? (détection **prise en charge** ; si 0, ignorez l'interrupteur 1) |
| 3 | L'énergie dépensée est-elle dans le message ? |
| 4 | Des intervalles RR sont-ils dans le message ? |
| 5-7 | Réservés pour plus tard (toujours à 0) |

**Little-endian = « petit bout d'abord ».** Un nombre sur 2 octets arrive avec l'octet des unités en premier, comme une date écrite jour/mois. Pour le lire, on retourne les deux octets : `2C 03` se lit `03 2C`, soit 3 paquets de 256 + 44 = **812**.

### ▶ OBLIGATOIRE : l'outil de décodage (le code est fourni)

**Exécutez la cellule suivante avant toutes les autres.** Elle contient le décodeur `decode_hrm`, déjà écrit pour vous : vous n'avez pas à le programmer ni à le lire.
Elle n'affiche rien : c'est normal. Si vous oubliez de l'exécuter, les cellules suivantes affichent « Le décodeur n'est pas chargé ».

In [1]:
# ▶ OBLIGATOIRE : exécutez cette cellule en premier. Elle contient le décodeur decode_hrm (déjà écrit pour vous).
def decode_hrm(trame_hex):
    """Décode une trame Heart Rate Measurement (0x2A37) écrite en hexadécimal.

    trame_hex : texte comme "16 4B 20 03", "16-4B-20-03" ou "0x164B2003".
    Renvoie un dictionnaire lisible. Lève ValueError si la trame est incohérente.
    """
    # 1. Nettoyage : on enlève 0x, espaces, tirets, deux-points, puis on convertit en octets
    texte = trame_hex.replace("0x", "").replace("0X", "")
    for sep in (" ", "-", ":", ","):
        texte = texte.replace(sep, "")
    octets = bytes.fromhex(texte)
    if len(octets) < 2:
        raise ValueError("Trame trop courte : il faut au moins le 1er octet (interrupteurs) et la FC")

    # 2. Octet de flags : on teste chaque bit avec un ET binaire (&)
    flags = octets[0]
    fc_sur_16_bits = bool(flags & 0b00000001)   # bit 0
    contact_detecte = bool(flags & 0b00000010)  # bit 1
    contact_gere = bool(flags & 0b00000100)     # bit 2
    energie_presente = bool(flags & 0b00001000) # bit 3
    rr_presents = bool(flags & 0b00010000)      # bit 4

    if not contact_gere:
        contact = "non pris en charge"
    elif contact_detecte:
        contact = "contact détecté"
    else:
        contact = "pris en charge, PAS de contact"

    # 3. Lecture des champs dans l'ordre, avec un curseur i
    i = 1
    if fc_sur_16_bits:
        fc = int.from_bytes(octets[i:i + 2], "little")  # little-endian
        i += 2
    else:
        fc = octets[i]
        i += 1

    energie_kj = None
    if energie_presente:
        energie_kj = int.from_bytes(octets[i:i + 2], "little")
        i += 2

    rr_s = []
    if rr_presents:
        reste = octets[i:]
        if len(reste) == 0 or len(reste) % 2 != 0:
            raise ValueError("Interrupteur 4 allumé (RR annoncés), mais les octets restants ne vont pas par 2")
        for j in range(0, len(reste), 2):
            brut = int.from_bytes(reste[j:j + 2], "little")
            rr_s.append(brut / 1024)  # unité 1/1024 s -> secondes
        i = len(octets)

    if i != len(octets):
        raise ValueError(f"{len(octets) - i} octet(s) en trop par rapport à ce qu'annoncent les interrupteurs")

    return {
        "trame": " ".join(f"{b:02X}" for b in octets),
        "flags_bin": f"{flags:08b}",
        "format_fc": "uint16" if fc_sur_16_bits else "uint8",
        "contact": contact,
        "fc_bpm": fc,
        "energie_kJ": energie_kj,
        "rr_ms": [round(r * 1000, 1) for r in rr_s],
        "fc_instantanee_bpm": [round(60 / r, 1) for r in rr_s],  # 60 / RR (en s)
        "bits_reserves_a_1": bool(flags & 0b11100000),
    }

### ▶ L'exemple du cours : `16 48 2C 03`

Exécutez. **Vous devez voir** : `'contact': 'contact détecté'`, `'fc_bpm': 72` et `'rr_ms': [793.0]` (0,793 s entre deux battements).

In [2]:
if "decode_hrm" not in globals():
    raise NameError("Le décodeur n'est pas chargé : exécutez d'abord la cellule « L'outil de décodage » (celle qui commence par def decode_hrm), puis relancez cette cellule.")
# ▶ Exécutez : l'exemple du cours 16 48 2C 03
decode_hrm("16 48 2C 03")

{'trame': '16 48 2C 03',
 'flags_bin': '00010110',
 'format_fc': 'uint8',
 'contact': 'contact détecté',
 'fc_bpm': 72,
 'energie_kJ': None,
 'rr_ms': [793.0],
 'fc_instantanee_bpm': [75.7],
 'bits_reserves_a_1': False}

### ▶ Les 6 trames décodées par le notebook

Vous en avez décodé 3 à la main (trames 3, 5 et 2). Le notebook décode les 6. Exécutez.

Comment lire les colonnes :
- `flags_bin` : les 8 interrupteurs du premier octet (le bit 0 est le plus à droite) ;
- `format_fc` : FC sur 1 octet (`uint8`) ou 2 octets (`uint16`) ;
- `contact` : le capteur touche-t-il la peau ? ;
- `fc_bpm` : la fréquence cardiaque, en battements par minute ;
- `energie_kJ` : l'énergie dépensée (`<NA>` = absente) ;
- `rr_ms` : le temps entre deux battements, en millisecondes (`[]` = absent) ;
- `fc_instantanee_bpm` : 60 divisé par le RR en secondes ;
- `bits_reserves_a_1` : un interrupteur réservé (5 à 7) est-il allumé ? (normalement `False`).

In [3]:
if "decode_hrm" not in globals():
    raise NameError("Le décodeur n'est pas chargé : exécutez d'abord la cellule « L'outil de décodage » (celle qui commence par def decode_hrm), puis relancez cette cellule.")
# ▶ Exécutez : les 6 trames de l'énoncé (étape 4), toutes décodées par le notebook
TRAMES = {
    1: "00 48",
    2: "01 5A 00",
    3: "06 3E",
    4: "0E 55 E8 03",
    5: "16 4B 20 03",
    6: "1E 4E C8 00 F0 02 10 03",
}

import pandas as pd
tableau = pd.DataFrame([{"n°": n, **decode_hrm(t)} for n, t in TRAMES.items()]).set_index("n°")
tableau["energie_kJ"] = tableau["energie_kJ"].astype("Int64")  # entier, vide si absent
tableau

,trame,flags_bin,format_fc,contact,fc_bpm,energie_kJ,rr_ms,fc_instantanee_bpm,bits_reserves_a_1
n°,,,,,,,,,
1,00 48,00000000,uint8,non pris en charge,72,<NA>,[],[],False
2,01 5A 00,00000001,uint16,non pris en charge,90,<NA>,[],[],False
3,06 3E,00000110,uint8,contact détecté,62,<NA>,[],[],False
4,0E 55 E8 03,00001110,uint8,contact détecté,85,1000,[],[],False
5,16 4B 20 03,00010110,uint8,contact détecté,75,<NA>,[781.2],[76.8],False
6,1E 4E C8 00 F0 02 10 03,00011110,uint8,contact détecté,78,200,"[734.4, 765.6]","[81.7, 78.4]",False


### ✏️ À vous : comparez avec vos réponses faites à la main

1. Dans la cellule suivante, remplacez `None` par la FC que vous avez trouvée à la main pour les trames **3, 5 et 2**.
2. Laissez `None` pour les trames 1, 4 et 6 (sauf si vous les avez faites en bonus).
3. Exécutez. **Vous devez voir** `OK` pour chaque trame faite à la main.

In [4]:
if "decode_hrm" not in globals():
    raise NameError("Le décodeur n'est pas chargé : exécutez d'abord la cellule « L'outil de décodage » (celle qui commence par def decode_hrm), puis relancez cette cellule.")
# ✏️ À VOUS — Remplacez None par la FC trouvée À LA MAIN pour les trames 3, 5 et 2.
# Trames 1, 4 et 6 : laissez None (le notebook les décode seul). La cellule fonctionne quand même.
MES_FC = {1: None, 2: 90, 3: 62, 4: None, 5: 75, 6: None}

for n, trame in TRAMES.items():
    attendu = decode_hrm(trame)["fc_bpm"]
    if MES_FC[n] is None:
        verdict = "non renseigné (pas faite à la main)"
    elif MES_FC[n] == attendu:
        verdict = "OK"
    else:
        verdict = f"à revoir (le décodeur trouve {attendu})"
    print(f"Trame {n} [{trame}] : votre FC = {MES_FC[n]} -> {verdict}")

Trame 1 [00 48] : votre FC = None -> non renseigné (pas faite à la main)
Trame 2 [01 5A 00] : votre FC = 90 -> OK
Trame 3 [06 3E] : votre FC = 62 -> OK
Trame 4 [0E 55 E8 03] : votre FC = None -> non renseigné (pas faite à la main)
Trame 5 [16 4B 20 03] : votre FC = 75 -> OK
Trame 6 [1E 4E C8 00 F0 02 10 03] : votre FC = None -> non renseigné (pas faite à la main)


## 2. ▶ Le piège du little-endian (« petit bout d'abord »)

La trame 2 (`01 5A 00`) a l'interrupteur 0 allumé : la FC prend **2 octets**, `5A 00`.
Le capteur envoie le petit bout d'abord : `5A` (les unités), puis `00` (les paquets de 256).

La cellule suivante compare :
- la **bonne** lecture (little-endian : on retourne les octets → `00 5A` = 90) ;
- la **mauvaise** lecture (big-endian : on lit dans l'ordre d'arrivée → `5A 00` = 90 × 256 = 23 040).

In [5]:
# ▶ Exécutez : les deux façons de lire les octets 5A 00
deux_octets = bytes.fromhex("5A00")
print("Lecture little-endian (correcte) :", int.from_bytes(deux_octets, "little"), "bpm")
print("Lecture big-endian (erreur)      :", int.from_bytes(deux_octets, "big"), "bpm")

Lecture little-endian (correcte) : 90 bpm
Lecture big-endian (erreur)      : 23040 bpm


## 3. ▶ Tests automatiques du décodeur (pas besoin de lire ce code)

Un décodeur qui ira dans un dispositif médical doit être **testé**. On vérifie deux choses :
- il trouve les bonnes valeurs sur les trames de l'exercice ;
- il **refuse** un message incomplet ou trop long, au lieu d'inventer un résultat.

Ce sera exigé au projet 2 (tests unitaires, cas limites). **Vous devez voir** à la fin : `Tous les tests sont passés.`

In [6]:
if "decode_hrm" not in globals():
    raise NameError("Le décodeur n'est pas chargé : exécutez d'abord la cellule « L'outil de décodage » (celle qui commence par def decode_hrm), puis relancez cette cellule.")
# ▶ Exécutez (pas besoin de lire ce code) : on vérifie que le décodeur donne les bonnes réponses.
# Valeurs attendues, calculées à la main à partir de la norme
assert decode_hrm("00 48")["fc_bpm"] == 72
assert decode_hrm("01 5A 00")["fc_bpm"] == 90
assert decode_hrm("06 3E")["contact"] == "contact détecté"
assert decode_hrm("04 3E")["contact"] == "pris en charge, PAS de contact"
assert decode_hrm("0E 55 E8 03")["energie_kJ"] == 1000
assert decode_hrm("16 4B 20 03")["rr_ms"] == [781.2]
assert decode_hrm("1E 4E C8 00 F0 02 10 03")["rr_ms"] == [734.4, 765.6]
assert decode_hrm("0x164B2003") == decode_hrm("16-4B-20-03")  # formats d'écriture tolérés

# Trames invalides : le décodeur doit refuser plutôt que d'inventer une valeur
for mauvaise in ["16", "10 48 2C", "00 48 99"]:
    try:
        decode_hrm(mauvaise)
        print("ÉCHEC : trame acceptée à tort :", mauvaise)
    except ValueError as erreur:
        print(f"Trame {mauvaise!r} refusée : {erreur}")
print("Tous les tests sont passés.")

Trame '16' refusée : Trame trop courte : il faut au moins le 1er octet (interrupteurs) et la FC
Trame '10 48 2C' refusée : Interrupteur 4 allumé (RR annoncés), mais les octets restants ne vont pas par 2
Trame '00 48 99' refusée : 1 octet(s) en trop par rapport à ce qu'annoncent les interrupteurs
Tous les tests sont passés.


## 4. ✏️ Questions (réponses courtes dans le fichier livrable)

1. Une FC humaine tient dans 1 octet (de 0 à 255). Pourquoi la norme prévoit-elle quand même une FC sur 2 octets ?
2. Trame `04 3E` : l'interrupteur 2 est allumé, l'interrupteur 1 est éteint. Que doit afficher une application de télésurveillance ? Faut-il envoyer la FC de 62 bpm au soignant ?
3. Dans la trame 6, pourquoi y a-t-il **deux** intervalles RR pour une seule FC ?
4. Trame 5 : si on lit les 2 octets du RR dans le mauvais ordre (big-endian, sans les retourner), que devient le RR ? Quel contrôle de bon sens aurait repéré l'erreur ?
5. Pour un dispositif médical, quel est l'intérêt d'utiliser ce format **normalisé** plutôt qu'un format inventé par l'équipe ?

### Réponses de l’équipe 02

1. **Pourquoi prévoir une FC sur deux octets ?** Un octet ne représente que 0 à 255 ; le format uint16 permet de coder des valeurs au-delà de 255 sans changer de caractéristique. Le bit 0 précise le format employé : 255 est une limite d’encodage, pas une limite physiologique universelle.

2. **Que faire de `04 3E` ?** Le capteur sait détecter le contact, mais indique son absence. L’application doit afficher « contact absent / mesure non fiable » et ne pas présenter les 62 bpm au soignant comme une mesure valide ; si la valeur est conservée ou transmise, elle doit rester explicitement marquée invalide.

3. **Pourquoi deux RR dans la trame 6 ?** Une notification peut regrouper plusieurs intervalles entre battements, du plus ancien au plus récent. La FC annoncée (78 bpm) est une valeur distincte, potentiellement lissée, qui ne remplace pas les deux RR de 734,375 et 765,625 ms.

4. **RR de la trame 5 lu en big-endian ?** `20 03` devient `0x2003` = 8195, soit 8195 / 1024 = **8,00293 s** (8002,93 ms), au lieu de 0,78125 s. Cela correspond à environ **7,50 bpm**, très incohérent avec les 75 bpm annoncés : la comparaison FC/RR et un contrôle de plausibilité auraient signalé l’erreur de lecture.

5. **Pourquoi un format normalisé ?** Des capteurs et des clients de fabricants différents peuvent interpréter les mêmes champs, unités et indicateurs de qualité. Cela facilite les tests et réduit les ambiguïtés d’intégration, sans garantir à lui seul la sécurité ou la validation du dispositif.


## 5. ▶ Livrable

Exécutez la cellule suivante : elle écrit le tableau des 6 trames dans `trames_decodees.csv`.
Déposez ce fichier dans `analyses/m2_tp3/` du repo de l'équipe, avec la capture d'écran nRF Connect.
En Colab : icône dossier à gauche → clic droit sur le fichier → *Télécharger*.

In [7]:
# ▶ Exécutez : écrit le fichier trames_decodees.csv (les 6 trames) pour le livrable
tableau.to_csv("trames_decodees.csv")
print("Fichier écrit : trames_decodees.csv (en Colab : icône dossier à gauche -> clic droit -> Télécharger)")

Fichier écrit : trames_decodees.csv (en Colab : icône dossier à gauche -> clic droit -> Télécharger)
